# Glioblastoma fluorescence analysis

Quantify per-channel fluorescence within a mask derived from the selected fluorescence channel from Olympus VSI scenes.
See `README.md` for installation, and data requirements.
Run from the repository root using the installed environment.


In [ ]:
from pathlib import Path
from glioblastoma_analysis.pipeline import analyze_slides


## Configure inputs

Set `INPUT_DIR` to the location of the `.vsi` files.
Channel indices are zero-based. Channel 1 (the second channel) was GFP in the original dataset
and remains the default. For your own dataset, verify the acquisition channel mapping and set
`MASK_CHANNEL` to the appropriate reference channel. With GFP as the reference, the mask is
GFP-derived and does not necessarily cover all tissue.
Outputs with matching names are overwritten; choose a new output folder for a separate run.


In [ ]:
INPUT_DIR = Path("data")
OUTPUT_DIR = Path("results")
MASK_CHANNEL = 1  # GFP in the original dataset; select the correct reference channel for your data.
FILENAMES = None  # All .vsi files, or an explicit list such as ["Image_01.vsi"] if only some of the images need to be run
SAVE_TIFFS = True
SAVE_CONTOURS = True
USE_SAVED_TIFFS = False  # True reuses exported TIFFs; original VSI files are still needed


## Segment and quantify

Gaussian smoothing (sigma 1), Otsu thresholding, removal of objects and holes smaller than
8,000 pixels, largest connected component, and binary closing with a disk of radius 5.
The mask is computed once per scene, then applied to every channel's original intensities.
Only the largest connected component is retained, which can exclude disconnected GFP-positive
regions when GFP is the reference channel. A scene with no detected region stops the run;
inspect masks before using the measurements.


In [ ]:
results = analyze_slides(
    INPUT_DIR, OUTPUT_DIR, filenames=FILENAMES, mask_channel=MASK_CHANNEL,
    save_tiffs=SAVE_TIFFS, save_contours=SAVE_CONTOURS,
    use_saved_tiffs=USE_SAVED_TIFFS,
)
results.head()


## Review outputs

`results.csv` contains one row per file, scene, and channel. Inspect the `*_MASK.png`
contours against the reference-channel signal, especially when a scene contains disconnected
GFP-positive regions. Measurements are raw intensities within the shared mask; this workflow
does not perform GFP/DAPI normalization.


In [ ]:
print(f"Saved {len(results)} measurements to {(OUTPUT_DIR / 'results.csv').resolve()}")
